In [3]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import RAW_DATA_DIR, PROCESSED_DATA_DIR, ID_COL

In [4]:
train = pd.read_csv(RAW_DATA_DIR / "application_train.csv", usecols=["SK_ID_CURR"])
test = pd.read_csv(RAW_DATA_DIR / "application_test.csv", usecols=["SK_ID_CURR"])

bureau = pd.read_csv(RAW_DATA_DIR / "bureau.csv")
bureau_balance = pd.read_csv(RAW_DATA_DIR / "bureau_balance.csv")

base = pd.concat([train, test], axis=0, ignore_index=True)
base = base.drop_duplicates(subset=["SK_ID_CURR"]).reset_index(drop=True)

print(train.shape, test.shape, base.shape)
print(bureau.shape, bureau_balance.shape)

(307511, 1) (48744, 1) (356255, 1)
(1716428, 17) (27299925, 3)


In [5]:
display(bureau.head())
display(bureau_balance.head())

print("bureau unique SK_ID_CURR:", bureau["SK_ID_CURR"].nunique())
print("bureau unique SK_ID_BUREAU:", bureau["SK_ID_BUREAU"].nunique())
print("bureau_balance unique SK_ID_BUREAU:", bureau_balance["SK_ID_BUREAU"].nunique())

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN


,SK_ID_BUREAU,MONTHS_BALANCE,STATUS
0,5715448,0,C
1,5715448,-1,C
2,5715448,-2,C
3,5715448,-3,C
4,5715448,-4,C


bureau unique SK_ID_CURR: 305811
bureau unique SK_ID_BUREAU: 1716428
bureau_balance unique SK_ID_BUREAU: 817395


In [7]:
bb = bureau_balance.copy()
status_dummies = pd.get_dummies(bb["STATUS"], prefix="BB_STATUS")
bb = pd.concat([bb, status_dummies], axis=1)

In [9]:
bb_agg = bb.groupby("SK_ID_BUREAU").agg(
    BB_MONTHS_MIN=("MONTHS_BALANCE", "min"),
    BB_MONTHS_MAX=("MONTHS_BALANCE", "max"),
    BB_MONTHS_COUNT=("MONTHS_BALANCE", "count"),
)

status_cols = [c for c in bb.columns if c.startswith("BB_STATUS_")]

bb_status_agg = bb.groupby("SK_ID_BUREAU")[status_cols].agg(["mean", "sum"])
bb_status_agg.columns = [
    f"{col}_{stat}".upper() for col, stat in bb_status_agg.columns
]
bb_status_agg = bb_status_agg.reset_index()

bb_agg = bb_agg.reset_index().merge(bb_status_agg, on="SK_ID_BUREAU", how="left")
bb_agg.head()

,SK_ID_BUREAU,BB_MONTHS_MIN,BB_MONTHS_MAX,BB_MONTHS_COUNT,BB_STATUS_0_MEAN,BB_STATUS_0_SUM,BB_STATUS_1_MEAN,BB_STATUS_1_SUM,BB_STATUS_2_MEAN,BB_STATUS_2_SUM,BB_STATUS_3_MEAN,BB_STATUS_3_SUM,BB_STATUS_4_MEAN,BB_STATUS_4_SUM,BB_STATUS_5_MEAN,BB_STATUS_5_SUM,BB_STATUS_C_MEAN,BB_STATUS_C_SUM,BB_STATUS_X_MEAN,BB_STATUS_X_SUM
0,5001709,-96,0,97,0.000000,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.886598,86,0.113402,11
1,5001710,-82,0,83,0.060241,5,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.578313,48,0.361446,30
2,5001711,-3,0,4,0.750000,3,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.000000,0,0.250000,1
3,5001712,-18,0,19,0.526316,10,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.473684,9,0.000000,0
4,5001713,-21,0,22,0.000000,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.000000,0,1.000000,22


In [10]:
bureau_full = bureau.merge(bb_agg, on="SK_ID_BUREAU", how="left")
print(bureau_full.shape)

(1716428, 36)


In [11]:
bureau_full["CREDIT_ACTIVE_IS_ACTIVE"] = (bureau_full["CREDIT_ACTIVE"] == "Active").astype(int)
bureau_full["CREDIT_ACTIVE_IS_CLOSED"] = (bureau_full["CREDIT_ACTIVE"] == "Closed").astype(int)

bureau_full["HAS_DEBT"] = (bureau_full["AMT_CREDIT_SUM_DEBT"].fillna(0) > 0).astype(int)
bureau_full["HAS_OVERDUE"] = (bureau_full["AMT_CREDIT_SUM_OVERDUE"].fillna(0) > 0).astype(int)

bureau_full["DEBT_CREDIT_RATIO"] = (
    bureau_full["AMT_CREDIT_SUM_DEBT"] / bureau_full["AMT_CREDIT_SUM"]
).replace([np.inf, -np.inf], np.nan)

bureau_full["OVERDUE_DEBT_RATIO"] = (
    bureau_full["AMT_CREDIT_SUM_OVERDUE"] / bureau_full["AMT_CREDIT_SUM_DEBT"]
).replace([np.inf, -np.inf], np.nan)

In [15]:
agg_dict = {
    "SK_ID_BUREAU": "count",
    "CREDIT_ACTIVE_IS_ACTIVE": "sum",
    "CREDIT_ACTIVE_IS_CLOSED": "sum",
    "HAS_DEBT": "sum",
    "HAS_OVERDUE": "sum",
    "DAYS_CREDIT": ["mean", "max"],
    "DAYS_CREDIT_UPDATE": "mean",
    "AMT_CREDIT_SUM": ["mean", "sum"],
    "AMT_CREDIT_SUM_DEBT": ["mean", "sum"],
    "AMT_CREDIT_SUM_OVERDUE": "max",
    "DEBT_CREDIT_RATIO": "mean",
    "OVERDUE_DEBT_RATIO": "mean",
    "BB_MONTHS_COUNT": ["mean", "max"],
}

bb_status_mean_cols = [
    c for c in bureau_full.columns
    if c.startswith("BB_STATUS_") and c.endswith("_MEAN")
]

bb_status_sum_cols = [
    c for c in bureau_full.columns
    if c.startswith("BB_STATUS_") and c.endswith("_SUM")
]

for col in bb_status_mean_cols:
    agg_dict[col] = "mean"

for col in bb_status_sum_cols:
    agg_dict[col] = "sum"

bureau_curr_agg = bureau_full.groupby("SK_ID_CURR").agg(agg_dict)

bureau_curr_agg.columns = [
    "BUREAU_" + "_".join(col).upper() if isinstance(col, tuple) else "BUREAU_" + col.upper()
    for col in bureau_curr_agg.columns
]

bureau_curr_agg = bureau_curr_agg.reset_index()

bureau_curr_agg["BUREAU_ACTIVE_RATIO"] = (
    bureau_curr_agg["BUREAU_CREDIT_ACTIVE_IS_ACTIVE_SUM"] /
    bureau_curr_agg["BUREAU_SK_ID_BUREAU_COUNT"]
)

bureau_curr_agg["BUREAU_CLOSED_RATIO"] = (
    bureau_curr_agg["BUREAU_CREDIT_ACTIVE_IS_CLOSED_SUM"] /
    bureau_curr_agg["BUREAU_SK_ID_BUREAU_COUNT"]
)

bureau_curr_agg["BUREAU_OVERDUE_RATIO"] = (
    bureau_curr_agg["BUREAU_HAS_OVERDUE_SUM"] /
    bureau_curr_agg["BUREAU_SK_ID_BUREAU_COUNT"]
)

In [16]:
bureau_features = base.merge(bureau_curr_agg, on="SK_ID_CURR", how="left")

print(bureau_features.shape)
print(bureau_features["SK_ID_CURR"].nunique())
display(bureau_features.head())
display(bureau_features.isnull().mean().sort_values(ascending=False).head(20))

(356255, 37)
356255


,SK_ID_CURR,BUREAU_SK_ID_BUREAU_COUNT,BUREAU_CREDIT_ACTIVE_IS_ACTIVE_SUM,BUREAU_CREDIT_ACTIVE_IS_CLOSED_SUM,BUREAU_HAS_DEBT_SUM,BUREAU_HAS_OVERDUE_SUM,BUREAU_DAYS_CREDIT_MEAN,BUREAU_DAYS_CREDIT_MAX,BUREAU_DAYS_CREDIT_UPDATE_MEAN,BUREAU_AMT_CREDIT_SUM_MEAN,...,BUREAU_BB_STATUS_1_SUM_SUM,BUREAU_BB_STATUS_2_SUM_SUM,BUREAU_BB_STATUS_3_SUM_SUM,BUREAU_BB_STATUS_4_SUM_SUM,BUREAU_BB_STATUS_5_SUM_SUM,BUREAU_BB_STATUS_C_SUM_SUM,BUREAU_BB_STATUS_X_SUM_SUM,BUREAU_ACTIVE_RATIO,BUREAU_CLOSED_RATIO,BUREAU_OVERDUE_RATIO
0,100002,8.0,2.0,6.0,1.0,0.0,-874.00,-103.0,-499.875,108131.945625,...,27.0,0.0,0.0,0.0,0.0,23.0,15.0,0.25,0.75,0.0
1,100003,4.0,1.0,3.0,0.0,0.0,-1400.75,-606.0,-816.000,254350.125000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.75,0.0
2,100004,2.0,0.0,2.0,0.0,0.0,-867.00,-408.0,-532.000,94518.900000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,1.00,0.0
3,100006,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,1.0,0.0,1.0,0.0,0.0,-1149.00,-1149.0,-783.000,146250.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,1.00,0.0


BUREAU_BB_STATUS_0_MEAN_MEAN       0.622344
BUREAU_BB_STATUS_C_MEAN_MEAN       0.622344
BUREAU_BB_STATUS_4_MEAN_MEAN       0.622344
BUREAU_BB_STATUS_3_MEAN_MEAN       0.622344
BUREAU_BB_STATUS_2_MEAN_MEAN       0.622344
BUREAU_BB_STATUS_1_MEAN_MEAN       0.622344
BUREAU_BB_STATUS_X_MEAN_MEAN       0.622344
BUREAU_BB_MONTHS_COUNT_MAX         0.622344
BUREAU_BB_MONTHS_COUNT_MEAN        0.622344
BUREAU_BB_STATUS_5_MEAN_MEAN       0.622344
BUREAU_OVERDUE_DEBT_RATIO_MEAN     0.391489
BUREAU_DEBT_CREDIT_RATIO_MEAN      0.169210
BUREAU_AMT_CREDIT_SUM_DEBT_MEAN    0.165095
BUREAU_AMT_CREDIT_SUM_MEAN         0.141601
BUREAU_BB_STATUS_2_SUM_SUM         0.141595
BUREAU_BB_STATUS_0_SUM_SUM         0.141595
BUREAU_BB_STATUS_3_SUM_SUM         0.141595
BUREAU_BB_STATUS_4_SUM_SUM         0.141595
BUREAU_BB_STATUS_5_SUM_SUM         0.141595
BUREAU_BB_STATUS_C_SUM_SUM         0.141595
dtype: float64

In [17]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
bureau_features.to_parquet(PROCESSED_DATA_DIR / "bureau_features.parquet", index=False)

In [18]:
from src.features_bureau import build_bureau_features

bureau_features = build_bureau_features(save=True)
print(bureau_features.shape)
display(bureau_features.head())

(356255, 37)


,SK_ID_CURR,BUREAU_SK_ID_BUREAU_COUNT,BUREAU_CREDIT_ACTIVE_IS_ACTIVE_SUM,BUREAU_CREDIT_ACTIVE_IS_CLOSED_SUM,BUREAU_HAS_DEBT_SUM,BUREAU_HAS_OVERDUE_SUM,BUREAU_DAYS_CREDIT_MEAN,BUREAU_DAYS_CREDIT_MAX,BUREAU_DAYS_CREDIT_UPDATE_MEAN,BUREAU_AMT_CREDIT_SUM_MEAN,...,BUREAU_BB_STATUS_1_SUM_SUM,BUREAU_BB_STATUS_2_SUM_SUM,BUREAU_BB_STATUS_3_SUM_SUM,BUREAU_BB_STATUS_4_SUM_SUM,BUREAU_BB_STATUS_5_SUM_SUM,BUREAU_BB_STATUS_C_SUM_SUM,BUREAU_BB_STATUS_X_SUM_SUM,BUREAU_ACTIVE_RATIO,BUREAU_CLOSED_RATIO,BUREAU_OVERDUE_RATIO
0,100002,8.0,2.0,6.0,1.0,0.0,-874.00,-103.0,-499.875,108131.945625,...,27.0,0.0,0.0,0.0,0.0,23.0,15.0,0.25,0.75,0.0
1,100003,4.0,1.0,3.0,0.0,0.0,-1400.75,-606.0,-816.000,254350.125000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.75,0.0
2,100004,2.0,0.0,2.0,0.0,0.0,-867.00,-408.0,-532.000,94518.900000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,1.00,0.0
3,100006,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,1.0,0.0,1.0,0.0,0.0,-1149.00,-1149.0,-783.000,146250.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,1.00,0.0
